# HSI Adapter Training

Навчальний notebook для HSI-адаптера ViPT: спочатку перевіряє PyTorch/CUDA, потім запускає короткий прогін на обмеженій кількості послідовностей, а повне тренування доступне окремим перемикачем.

Це експериментальний HSI шлях. RGB-T checkpoint ViPT використовується лише як ініціалізація, а не як готова HSI модель. Запускайте notebook з кореня репозиторію; у Colab спершу відкрийте/клонуйте весь репозиторій та підготуйте доступ до датасету. Не встановлюйте CPU build PyTorch поверх CUDA runtime.

In [1]:
import os
import random
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "hsi_tracking").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Repository root not found. Open this notebook from the repository checkout."
    )
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np

SEED = 42
SENSOR = "VIS"
DATA_ROOT = PROJECT_ROOT / "data"
CHECKPOINT = PROJECT_ROOT / "third_party" / "ViPT" / "models" / "ViPT_deep_rgbt.pth"
OUTPUT_DIR = PROJECT_ROOT / "results" / "full-model" / f"{DATA_ROOT.name}_{SENSOR}" / "training"

BATCH_SIZE = 1
EPOCHS = 5
NUM_WORKERS = 0
INPUT_SCALE = 255.0
MIXED_PRECISION = True
DEBUG_MODE = True
QUICK_MAX_SEQUENCES = 1
QUICK_MAX_STEPS = 10
RUN_FULL_TRAINING = False

random.seed(SEED)
np.random.seed(SEED)
print(f"Project: {PROJECT_ROOT}")
print(f"Data: {DATA_ROOT}")
print(f"Checkpoint: {CHECKPOINT}")
print(f"Output: {OUTPUT_DIR}")

Project: /home/bohdan/3_course/AI-3-course/PROJECT-hyper-spectral-object-tracking/hyper-spectral-object-tracking
Data: /home/bohdan/3_course/AI-3-course/PROJECT-hyper-spectral-object-tracking/hyper-spectral-object-tracking/data
Checkpoint: /home/bohdan/3_course/AI-3-course/PROJECT-hyper-spectral-object-tracking/hyper-spectral-object-tracking/third_party/ViPT/models/ViPT_deep_rgbt.pth
Output: /home/bohdan/3_course/AI-3-course/PROJECT-hyper-spectral-object-tracking/hyper-spectral-object-tracking/results/full-model/data_VIS/training


## GPU/CUDA та середовище

Перевірка не змінює встановлений PyTorch. Якщо CUDA недоступна, ноутбук автоматично працюватиме на CPU; CPU годиться для smoke test, але повний train може бути дуже повільним.

In [2]:
try:
    import torch
except ImportError as error:
    raise RuntimeError(
        "PyTorch is missing in this notebook kernel. Select the vipt environment or a CUDA-enabled Colab runtime."
    ) from error

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")
print(f"cuDNN: {torch.backends.cudnn.version()}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    print(f"GPU memory: {gpu.total_memory / 1024**3:.2f} GiB")
    try:
        smoke = torch.randn((1024, 1024), device=DEVICE)
        _ = smoke @ smoke
        torch.cuda.synchronize()
        print("CUDA tensor smoke test: passed")
    except RuntimeError as error:
        print(f"CUDA smoke test failed; falling back to CPU: {error}")
        DEVICE = torch.device("cpu")
else:
    DEVICE = torch.device("cpu")
    print("CUDA is not available; selected CPU")

if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"ViPT initialization checkpoint not found: {CHECKPOINT}")
print(f"Selected device: {DEVICE}")
print(f"AMP requested: {MIXED_PRECISION and DEVICE.type == 'cuda'}")
print("Note: the current train_hsi_adapters implementation does not yet use AMP.")

Python: 3.7.12
PyTorch: 1.7.0+cpu
CUDA build: None
cuDNN: None
CUDA is not available; selected CPU
Selected device: cpu
AMP requested: False
Note: the current train_hsi_adapters implementation does not yet use AMP.


/home/bohdan/miniforge3/envs/vipt/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Швидкий піднабір послідовностей

Вибираються перші `QUICK_MAX_SEQUENCES` послідовності у стабільному алфавітному порядку. Створюється окрема папка з symlink на файли, без дублювання великих HSI кадрів. Для одного sequence folder код також працює.

In [ ]:
import tempfile

from hsi_tracking.training import discover_training_sequences

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_ROOT}")
ALL_SEQUENCE_DIRS = discover_training_sequences(DATA_ROOT)
if not ALL_SEQUENCE_DIRS:
    raise RuntimeError(f"No labeled HSI sequences found under {DATA_ROOT}")

QUICK_SEQUENCE_DIRS = ALL_SEQUENCE_DIRS[:QUICK_MAX_SEQUENCES]
DEBUG_SUBSET_TEMP = tempfile.TemporaryDirectory(prefix="hsi_debug_subset_")
DEBUG_DATA_ROOT = Path(DEBUG_SUBSET_TEMP.name)
for index, sequence_dir in enumerate(QUICK_SEQUENCE_DIRS):
    subset_sequence = DEBUG_DATA_ROOT / f"{index:03d}_{sequence_dir.name}"
    subset_sequence.mkdir(parents=True)
    for source in sequence_dir.iterdir():
        target = subset_sequence / source.name
        os.symlink(source.resolve(), target, target_is_directory=source.is_dir())

subset_frame_counts = {
    path.name: sum(frame.suffix.lower() in {".png", ".tif", ".tiff"} for frame in path.iterdir())
    for path in DEBUG_DATA_ROOT.iterdir()
}
print(f"Labeled sequences available: {len(ALL_SEQUENCE_DIRS)}")
print(f"Selected for quick run: {len(QUICK_SEQUENCE_DIRS)}")
print(f"Subset folder: {DEBUG_DATA_ROOT}")
print(f"HSI frame counts: {subset_frame_counts}")
print("Tracking data has one target per sequence; classification class counts do not apply.")
print("Selected sequences:")
for sequence_dir in QUICK_SEQUENCE_DIRS:
    print(f"  {sequence_dir}")

## Dataset і DataLoader

`train_hsi_adapters()` будує `HSIFramePairDataset` з сусідніх кадрів і GT box-ів, а потім DataLoader з налаштованими `BATCH_SIZE` та `NUM_WORKERS`. Кадри читаються під час ітерації; повні HSI куби не завантажуються всі одночасно. Аугментації/кропи для template та search створюються штатною ViPT processing-функцією.

## Модель, loss і optimizer

Trainer завантажує ViPT RGB-T checkpoint як початкові ваги, заморожує backbone/head і навчає лише spectral projector та CNN prompt adapter (поточна перевірка бюджету має залишати trainable-параметри нижче 3%). Внутрішньо використовуються AdamW, SmoothL1 + GIoU loss та gradient clipping. `MIXED_PRECISION` поки лише фіксує намір: AMP ще не підключений до trainer.

## Quick smoke run

Зараз буде лише 1 епоха, максимум `QUICK_MAX_STEPS` оновлень і обмежена кількість послідовностей. Це перевірка сумісності data → model → backward → checkpoint, а не оцінка якості навчання.

In [ ]:
from hsi_tracking.training import train_hsi_adapters

SMOKE_OUTPUT_DIR = OUTPUT_DIR / "smoke"
smoke_history = train_hsi_adapters(
    data_root=DEBUG_DATA_ROOT,
    sensor=SENSOR,
    base_checkpoint=CHECKPOINT,
    output_dir=SMOKE_OUTPUT_DIR,
    epochs=1,
    batch_size=BATCH_SIZE,
    max_sequences=QUICK_MAX_SEQUENCES,
    max_steps=QUICK_MAX_STEPS,
    input_scale=INPUT_SCALE,
    num_workers=NUM_WORKERS,
    device=DEVICE,
)
print(smoke_history)
print((SMOKE_OUTPUT_DIR / "training_history.json").read_text())

## Повне тренування на всіх HSI sequence

Перед запуском перевірте `DATA_ROOT` і список `ALL_SEQUENCE_DIRS` вище. Встановіть `DEBUG_MODE = False` та `RUN_FULL_TRAINING = True`; повний прогін бере всі HSI sequences, що знаходяться під root, без обмеження кроків. У поточному checkout це одна HSI sequence `automobile4` (201 кадр); `ball` і `doll3` є JPG, тому не потрапляють до HSI trainer.

In [ ]:
if RUN_FULL_TRAINING:
    if DEBUG_MODE:
        raise RuntimeError("Set DEBUG_MODE = False before enabling full training.")
    full_history = train_hsi_adapters(
        data_root=DATA_ROOT,
        sensor=SENSOR,
        base_checkpoint=CHECKPOINT,
        output_dir=OUTPUT_DIR / "full",
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        max_sequences=None,
        max_steps=0,
        input_scale=INPUT_SCALE,
        num_workers=NUM_WORKERS,
        device=DEVICE,
    )
    print(full_history)
else:
    print("Full training is disabled. Set DEBUG_MODE=False and RUN_FULL_TRAINING=True to enable.")

## Логи та checkpoints

Trainer записує `adapter_latest.pth` і `training_history.json` окремо для smoke/full. Поточна реалізація не має best-checkpoint за validation, scheduler, early stopping чи resume optimizer state; history містить train loss і trainable-параметри. Нижче відобразяться доступні артефакти.

In [ ]:
import json

artifact_dirs = [OUTPUT_DIR / "smoke", OUTPUT_DIR / "full"]
for artifact_dir in artifact_dirs:
    checkpoint_path = artifact_dir / "adapter_latest.pth"
    history_path = artifact_dir / "training_history.json"
    if checkpoint_path.is_file():
        print(f"Checkpoint: {checkpoint_path} ({checkpoint_path.stat().st_size / 1024**2:.1f} MiB)")
    if history_path.is_file():
        records = json.loads(history_path.read_text())
        for record in records:
            print(
                f"{artifact_dir.name}: epoch={record['epoch']} steps={record['steps']} "
                f"mean_loss={record['mean_loss']} "
                f"trainable={record['trainable_parameter_ratio']:.2%}"
            )

if DEVICE.type == "cuda":
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")

## Швидка inference-перевірка

Перевіряє кілька кадрів першої вибраної sequence та друкує bbox, confidence і IoU з GT. Для smoke checkpoint це перевірка wiring на тренувальній sequence, не held-out validation і не доказ якості моделі.

In [ ]:
from hsi_tracking.data.dataset import load_hsi_sequence
from hsi_tracking.models.tracker import HSISequenceTracker, build_hsi_adapted_vipt

smoke_checkpoint = OUTPUT_DIR / "smoke" / "adapter_latest.pth"
full_checkpoint = OUTPUT_DIR / "full" / "adapter_latest.pth"
ADAPTER_CHECKPOINT = full_checkpoint if full_checkpoint.is_file() else smoke_checkpoint

if ADAPTER_CHECKPOINT.is_file():
    sequence = load_hsi_sequence(QUICK_SEQUENCE_DIRS[0], SENSOR)
    if sequence.annotation_kind != "groundtruth":
        raise ValueError("Inference comparison requires groundtruth_rect.txt")

    model, cfg = build_hsi_adapted_vipt(
        sensor=SENSOR,
        checkpoint=CHECKPOINT,
        device=DEVICE,
        input_scale=INPUT_SCALE,
    )
    model.load_adapters(ADAPTER_CHECKPOINT)
    tracker = HSISequenceTracker(model, cfg, device=DEVICE)
    tracker.initialize(sequence.load_cube(0), sequence.boxes[0])

    frame_limit = min(len(sequence), 4)
    reports = []
    for frame_index in range(1, frame_limit):
        prediction = tracker.track(sequence.load_cube(frame_index))
        predicted_box = prediction["target_bbox"]
        ground_truth = sequence.boxes[frame_index]
        left = max(predicted_box[0], ground_truth[0])
        top = max(predicted_box[1], ground_truth[1])
        right = min(predicted_box[0] + predicted_box[2], ground_truth[0] + ground_truth[2])
        bottom = min(predicted_box[1] + predicted_box[3], ground_truth[1] + ground_truth[3])
        intersection = max(0.0, right - left) * max(0.0, bottom - top)
        union = predicted_box[2] * predicted_box[3] + ground_truth[2] * ground_truth[3] - intersection
        iou = intersection / union if union > 0 else 0.0
        reports.append({"frame": frame_index, "bbox": predicted_box,
                        "confidence": prediction["best_score"], "iou": iou})

    for report in reports:
        print(report)
    if reports:
        print(f"Mean IoU over {len(reports)} frames: {np.mean([r['iou'] for r in reports]):.4f}")
else:
    print("No adapter checkpoint found. Run the smoke or full-training cell first.")

## Обмеження та запуск у Colab

У цьому checkout зараз є одна HSI sequence з GT (`automobile4`, 201 кадр). `ball` і `doll3` містять JPG, а не HSI mosaics, тому не входять до train. Щоб тренувати на кількох відео, додайте їхні HSI sequence folders із `groundtruth_rect.txt` у `DATA_ROOT`. Smoke режим бере лише першу HSI sequence та перші 10 optimizer steps.

У Colab відкрийте повний checkout репозиторію, підготуйте `data/` з усіма потрібними HSI sequence folders та забезпечте наявність `third_party/ViPT` і `ViPT_deep_rgbt.pth`. Встановіть залежності з `requirements-vipt-colab.txt`, зберігши CUDA-сумісний PyTorch, і запустіть потрібний patcher для ViPT. Результати пишуться в `results/full-model/data_VIS/training/`.

У поточного trainer немає validation loop, accuracy/F1, scheduler, early stopping, resume optimizer state чи AMP. Він записує train loss і adapter checkpoint; повне тренування тут не слід вважати якісно підтвердженим без окремої validation оцінки.